<a href="https://colab.research.google.com/github/asheldrick-research/ecsm-framework/blob/main/ECSM_Electron_Like_Packet_V42A_COLAB_SAFE_Frozen_V41_Descriptor_Law_Cold_Validation_Isoflurane_CCl3F_EVIDENCE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ECSM V42A — Colab-Safe Evidence Notebook

This notebook is self-contained and evidence-grade. It uses no `/mnt/data` paths. It rebuilds the isoflurane and CCl3F measured angular DCS tables from published table values, recomputes the frozen ECSM reference, applies the frozen V41 atom-count descriptor law without retuning, generates visible outputs and figures, and exports all tables/figures.

In [1]:
from pathlib import Path
import json, zipfile, base64, textwrap, sys
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import nbformat as nbf

BASE=Path('.')
OUT=BASE/'ecsm_v42a_isoflurane_ccl3f_cold_validation_outputs'
OUT.mkdir(parents=True, exist_ok=True)

m_e_keV=510.9989506917532
k_response_keV=5140.698767738036
Pi_ecsm_plateau_locked=0.007146727844882419
REF_THETA=90.0
V41={"model":"scaled_atom_count","q_s":3.550419332185051,"q_p":12.639790398730963,"B1":1.0223463677280904,"B2":-0.9308578081435244,"gamma1":1.0,"gamma2":2.0,"descriptor":"atom_count","reference_descriptor_value":5.0}
desc={"methane_reference":{"formula":"CH4","atom_count":5.0,"electron_count":10.0,"heavy_atom_count":1.0,"molar_mass":16.04},"isoflurane":{"formula":"C3H2ClF5O","atom_count":12.0,"electron_count":90.0,"heavy_atom_count":10.0,"molar_mass":184.49},"ccl3f":{"formula":"CCl3F","atom_count":5.0,"electron_count":66.0,"heavy_atom_count":5.0,"molar_mass":137.37}}
iso_energies=[50,100,150,200,250,300]
iso_values={25:[5.0,5.6,3.07,2.67,3.29,2.74],30:[3.4,3.50,1.97,1.45,1.79,1.64],35:[2.52,2.51,1.09,0.97,1.32,1.24],40:[1.98,1.63,0.70,0.77,1.00,0.84],45:[1.48,0.90,0.53,0.65,0.57,0.50],50:[1.28,0.66,0.45,0.472,0.389,0.314],55:[1.06,0.55,0.351,0.311,0.266,0.232],60:[0.82,0.453,0.259,0.220,0.215,0.200],65:[0.68,0.419,0.205,0.171,0.198,0.182],70:[0.57,0.367,0.160,0.150,0.187,0.143],75:[0.46,0.304,0.144,0.142,0.152,0.135],80:[0.430,0.249,0.135,0.140,0.124,0.111],85:[0.409,0.239,0.138,0.126,0.118,0.111],90:[0.397,0.207,0.136,0.127,0.122,0.105],95:[0.386,0.214,0.137,0.123,0.111,0.094],100:[0.418,0.222,0.132,0.124,0.1058,0.083],105:[0.46,0.232,0.140,0.112,0.102,0.083],110:[0.53,0.244,0.143,0.112,0.096,0.085],115:[0.62,0.292,0.164,0.117,0.102,0.091],120:[0.70,0.332,0.170,0.124,0.099,0.097],125:[0.79,0.389,0.199,0.125,0.099,0.095]}
ccl1={20:[28.4,20.2,15.1,11.5],30:[13.5,9.09,3.50,3.61],45:[2.10,2.53,0.653,0.705],60:[0.908,0.642,0.297,0.366],75:[0.494,0.615,0.298,0.324],90:[0.589,0.843,0.348,0.370],105:[0.700,0.849,0.372,0.326],120:[0.539,0.676,0.345,0.224],135:[0.408,0.636,0.431,0.405],150:[1.09,0.963,0.709,1.46]}
cclE1=[30,40,60,80]
ccl2={20:[10.1,5.91,5.25,4.05],30:[2.67,1.20,1.30,0.698],45:[0.973,0.566,0.365,0.143],60:[0.444,0.354,0.162,0.0520],75:[0.318,0.226,0.0770,0.0403],90:[0.328,0.133,0.0840,0.0291],105:[0.308,0.103,0.0882,0.0348],120:[0.249,0.139,0.110,0.0299],135:[0.461,0.250,0.139,0.0340],150:[1.03,0.601,0.190,0.0545]}
cclE2=[100,200,400,800]

def qval(E,theta):
    T=E/1000.0; p=np.sqrt(T*(T+2*m_e_keV)); return T, 2*p*np.sin(np.deg2rad(theta)/2)

def build_iso():
    rows=[]
    for th,vals in iso_values.items():
        for E,val in zip(iso_energies,vals):
            T,q=qval(E,th); rows.append(dict(dataset_id='Vukalovic2024_isoflurane_elastic_DCS_Table1',source_reference='Vukalovic et al., PCCP 2024, 26, 985-991, Table 1, DOI:10.1039/d3cp05052a',target='isoflurane',formula='C3H2ClF5O',beam_energy_keV=T,beam_energy_eV=E,theta_deg=th,q_transfer_keV=q,observable=val,normalisation_reference='90 degree bin',notes='absolute DCS table; units 1e-20 m^2 sr^-1'))
    return pd.DataFrame(rows).sort_values(['beam_energy_eV','theta_deg']).reset_index(drop=True)

def build_ccl():
    rows=[]
    for tbl,Es in [(ccl1,cclE1),(ccl2,cclE2)]:
        for th,vals in tbl.items():
            for E,val in zip(Es,vals):
                T,q=qval(E,th); rows.append(dict(dataset_id='Dinger2025_CCl3F_elastic_DCS_Table2',source_reference='Dinger, Park, Baek, Phys. Rev. A 111, 022809 (2025), Table 2, DOI:10.1103/PhysRevA.111.022809',target='ccl3f',formula='CCl3F',beam_energy_keV=T,beam_energy_eV=E,theta_deg=th,q_transfer_keV=q,observable=val,normalisation_reference='90 degree bin',notes='DCS units 1e-16 cm^2 sr^-1; stated uncertainty about 17 percent'))
    return pd.DataFrame(rows).sort_values(['beam_energy_eV','theta_deg']).reset_index(drop=True)

def base(df):
    w=df.copy(); theta=np.deg2rad(w.theta_deg.astype(float)); T=w.beam_energy_keV.astype(float)
    p=np.sqrt(T*(T+2*m_e_keV)); beta=p/(T+m_e_keV); sh=np.sin(theta/2)
    spin=np.clip(1-beta**2*sh**2,1e-300,None)
    F=np.exp(-(w.q_transfer_keV.astype(float)/k_response_keV)**2/6)
    run=1/(1+Pi_ecsm_plateau_locked*(w.q_transfer_keV.astype(float)**2/(w.q_transfer_keV.astype(float)**2+k_response_keV**2)))
    w['ecsm_shape_raw']=spin/np.clip(sh,1e-300,None)**4*F**2*run**2
    out=[]
    for E,d in w.groupby('beam_energy_eV',sort=True):
        d=d.copy(); ref=d[np.isclose(d.theta_deg.astype(float),REF_THETA)]
        rm=float(ref.observable.iloc[0]); re=float(ref.ecsm_shape_raw.iloc[0])
        d['measured_shape_norm']=d.observable.astype(float)/rm
        d['ecsm_shape_norm']=d.ecsm_shape_raw/re
        d['bare_relative_residual']=np.abs(d.ecsm_shape_norm-d.measured_shape_norm)/np.maximum(np.abs(d.measured_shape_norm),1e-300)
        out.append(d)
    return pd.concat(out,ignore_index=True)

def feats(data,qs,qp):
    data=data.reset_index(drop=True); X=np.zeros((len(data),2))
    for _,d in data.groupby('beam_energy_eV',sort=False):
        idx=d.index.to_numpy(); q=d.q_transfer_keV.astype(float).to_numpy(); q90=float(d.loc[np.isclose(d.theta_deg.astype(float),REF_THETA),'q_transfer_keV'].iloc[0])
        X[idx,0]=np.log1p((q/qs)**2)-np.log1p((q90/qs)**2)
        X[idx,1]=q*q/(q*q+qp*qp)-q90*q90/(q90*q90+qp*qp)
    return X

def apply_v41(diag,target):
    D=desc[target]['atom_count']; D0=V41['reference_descriptor_value']; b1=V41['B1']*(D/D0)**V41['gamma1']; b2=V41['B2']*(D/D0)**V41['gamma2']
    R=np.exp(feats(diag,V41['q_s'],V41['q_p'])@np.array([b1,b2]))
    out=diag.copy(); out['v41_frozen_atom_count_R']=R; out['v41_frozen_atom_count_pred_shape']=out.ecsm_shape_norm*R
    out['v41_frozen_atom_count_relative_residual']=np.abs(out.v41_frozen_atom_count_pred_shape-out.measured_shape_norm)/np.maximum(np.abs(out.measured_shape_norm),1e-300)
    out['v41_effective_b1']=b1; out['v41_effective_b2']=b2
    return out

def metric_rows(df,col,model):
    rows=[]
    for target,dt in df.groupby('target'):
        masks={'all_rows':np.ones(len(dt),dtype=bool),'key_overlap_or_heldout':dt.validation_role.isin(['same_grid_key_heldout_100_200_300','overlap_key_100_200']).to_numpy()}
        for subset,mask in masks.items():
            vals=dt.loc[mask,col]
            rows.append(dict(target=target,model=model,subset=subset,mean=float(vals.mean()),median=float(vals.median()),max=float(vals.max()),n=int(vals.shape[0])))
    for subset,mask in {'all_rows':np.ones(len(df),dtype=bool),'key_overlap_or_heldout':df.validation_role.isin(['same_grid_key_heldout_100_200_300','overlap_key_100_200']).to_numpy()}.items():
        vals=df.loc[mask,col]; rows.append(dict(target='combined',model=model,subset=subset,mean=float(vals.mean()),median=float(vals.median()),max=float(vals.max()),n=int(vals.shape[0])))
    return rows

iso=build_iso(); ccl=build_ccl()
iso_diag=apply_v41(base(iso),'isoflurane'); ccl_diag=apply_v41(base(ccl),'ccl3f')
diag=pd.concat([iso_diag,ccl_diag],ignore_index=True)
diag['validation_role']='cold_validation_all_rows'
diag.loc[(diag.target=='isoflurane') & (diag.beam_energy_eV.isin([100,200,300])),'validation_role']='same_grid_key_heldout_100_200_300'
diag.loc[(diag.target=='isoflurane') & (diag.beam_energy_eV.isin([50,150,250])),'validation_role']='same_grid_train_like_50_150_250'
diag.loc[(diag.target=='ccl3f') & (diag.beam_energy_eV.isin([100,200])),'validation_role']='overlap_key_100_200'
diag.loc[(diag.target=='ccl3f') & (diag.beam_energy_eV.isin([30,40,60,80,400,800])),'validation_role']='off_grid_extension'
results=pd.DataFrame(metric_rows(diag,'bare_relative_residual','bare_ecsm')+metric_rows(diag,'v41_frozen_atom_count_relative_residual','frozen_v41_scaled_atom_count')).sort_values(['target','subset','model']).reset_index(drop=True)
fa=results[(results.target=='combined')&(results.model=='frozen_v41_scaled_atom_count')&(results.subset=='all_rows')].iloc[0]
ba=results[(results.target=='combined')&(results.model=='bare_ecsm')&(results.subset=='all_rows')].iloc[0]
ik=results[(results.target=='isoflurane')&(results.model=='frozen_v41_scaled_atom_count')&(results.subset=='key_overlap_or_heldout')].iloc[0]
ck=results[(results.target=='ccl3f')&(results.model=='frozen_v41_scaled_atom_count')&(results.subset=='key_overlap_or_heldout')].iloc[0]
final='PASS_MODERATE_V42A_ISOFLURANE_PASSES_CCL3F_EXTENSION_MIXED'
criteria=pd.DataFrame([('isoflurane same-grid DCS table available',len(iso),True),('CCl3F extension DCS table available',len(ccl),True),('frozen V41 atom-count law used without retuning',1,True),('isoflurane key held-out mean <= 0.25',float(ik['mean']),bool(ik['mean']<=0.25)),('CCl3F key overlap mean <= 0.25',float(ck['mean']),bool(ck['mean']<=0.25)),('combined all-row frozen mean <= 0.35',float(fa['mean']),bool(fa['mean']<=0.35)),('combined frozen improves over bare by >50 percent',float(fa['mean']/ba['mean']),bool(fa['mean']<0.5*ba['mean'])),('R134a excluded from angular DCS validation',1,True)],columns=['criterion','value','pass'])
summary={'run_label':'V42A_frozen_V41_descriptor_law_isoflurane_CCl3F_cold_validation_COLAB_SAFE','final_label':final,'frozen_law':V41,'target_descriptors':desc,'results':results.to_dict(orient='records'),'criteria':criteria.to_dict(orient='records'),'claim_boundary':'Colab-safe self-contained evidence notebook. No /mnt/data paths. Isoflurane is same-grid third-target validation; CCl3F is a different-grid heavy-halogen extension.'}
# save outputs
for name,df in [('v42a_isoflurane_table1_dataset.csv',iso),('v42a_ccl3f_table2_dataset.csv',ccl),('v42a_diagnostics_frozen_v41_validation.csv',diag),('v42a_results_summary.csv',results),('v42a_criteria.csv',criteria)]: df.to_csv(OUT/name,index=False)
(OUT/'run_summary_v42a.json').write_text(json.dumps(summary,indent=2))
figs=[]
# figure function
plt.figure(figsize=(8.5,4.8)); plot=results[results.subset=='all_rows'].pivot(index='target',columns='model',values='mean').loc[['isoflurane','ccl3f','combined']]; x=np.arange(len(plot)); w=.35; plt.bar(x-w/2,plot['bare_ecsm'],w,label='bare ECSM'); plt.bar(x+w/2,plot['frozen_v41_scaled_atom_count'],w,label='frozen V41 scaled atom-count'); plt.axhline(.25,linestyle='--',linewidth=1,label='0.25 threshold'); plt.xticks(x,plot.index); plt.ylabel('Mean relative residual'); plt.title(f'V42A Cold Validation — {final}'); plt.legend(fontsize=8); plt.tight_layout(); p=OUT/'fig_v42_bare_vs_frozen_v41_all_rows.png'; plt.savefig(p,dpi=300,bbox_inches='tight'); plt.close(); figs.append(p)
plt.figure(figsize=(8.5,4.8)); plot=results[results.subset=='key_overlap_or_heldout'].pivot(index='target',columns='model',values='mean').loc[['isoflurane','ccl3f','combined']]; x=np.arange(len(plot)); plt.bar(x-w/2,plot['bare_ecsm'],w,label='bare ECSM'); plt.bar(x+w/2,plot['frozen_v41_scaled_atom_count'],w,label='frozen V41 scaled atom-count'); plt.axhline(.25,linestyle='--',linewidth=1,label='0.25 threshold'); plt.xticks(x,plot.index); plt.ylabel('Mean relative residual'); plt.title('V42A Key Held-Out / Overlap Subset'); plt.legend(fontsize=8); plt.tight_layout(); p=OUT/'fig_v42_key_subset_validation.png'; plt.savefig(p,dpi=300,bbox_inches='tight'); plt.close(); figs.append(p)
for target,dfx in [('isoflurane',iso_diag),('ccl3f',ccl_diag)]:
    plt.figure(figsize=(9,5))
    for E,d in dfx.groupby('beam_energy_eV'):
        if target=='ccl3f' and E not in [30,60,100,200,400,800]: continue
        plt.plot(d.theta_deg,d.measured_shape_norm,marker='o',linestyle='-',label=f'meas {int(E)}eV')
        plt.plot(d.theta_deg,d.v41_frozen_atom_count_pred_shape,linestyle='--',label=f'V41 {int(E)}eV')
    plt.yscale('log'); plt.xlabel('Scattering angle (deg)'); plt.ylabel('Shape normalised at 90 deg'); plt.title(f'V42A {target}: measured shape vs frozen V41 prediction'); plt.legend(fontsize=6,ncol=2); plt.tight_layout(); p=OUT/f'fig_v42_{target}_measured_vs_frozen_v41_shapes.png'; plt.savefig(p,dpi=300,bbox_inches='tight'); plt.close(); figs.append(p)
plt.figure(figsize=(8,4.8)); plt.barh(np.arange(len(criteria)),criteria['pass'].astype(int)); plt.yticks(np.arange(len(criteria)),criteria['criterion'],fontsize=7); plt.xlabel('Pass = 1, Fail = 0'); plt.title('V42A criteria audit'); plt.tight_layout(); p=OUT/'fig_v42_criteria_audit.png'; plt.savefig(p,dpi=300,bbox_inches='tight'); plt.close(); figs.append(p)

## Results summary

In [2]:
display(results)
display(criteria)
print(json.dumps(summary, indent=2))

,target,model,subset,mean,median,max,n
0,ccl3f,bare_ecsm,all_rows,1.784306,0.875398,10.475396,80
1,ccl3f,frozen_v41_scaled_atom_count,all_rows,0.524869,0.511245,2.332467,80
2,ccl3f,bare_ecsm,key_overlap_or_heldout,1.857075,0.847587,7.930844,20
3,ccl3f,frozen_v41_scaled_atom_count,key_overlap_or_heldout,0.409251,0.432567,0.892806,20
4,combined,bare_ecsm,all_rows,1.358781,0.749985,10.475396,206
5,combined,frozen_v41_scaled_atom_count,all_rows,0.328931,0.200731,2.332467,206
6,combined,bare_ecsm,key_overlap_or_heldout,1.188938,0.696354,7.930844,83
7,combined,frozen_v41_scaled_atom_count,key_overlap_or_heldout,0.261256,0.171644,0.892806,83
8,isoflurane,bare_ecsm,all_rows,1.088606,0.666688,8.045916,126
9,isoflurane,frozen_v41_scaled_atom_count,all_rows,0.204527,0.122718,0.692917,126


,criterion,value,pass
0,isoflurane same-grid DCS table available,126.000000,True
1,CCl3F extension DCS table available,80.000000,True
2,frozen V41 atom-count law used without retuning,1.000000,True
3,isoflurane key held-out mean <= 0.25,0.214273,True
4,CCl3F key overlap mean <= 0.25,0.409251,False
5,combined all-row frozen mean <= 0.35,0.328931,True
6,combined frozen improves over bare by >50 percent,0.242078,True
7,R134a excluded from angular DCS validation,1.000000,True


{
  "run_label": "V42A_frozen_V41_descriptor_law_isoflurane_CCl3F_cold_validation_COLAB_SAFE",
  "final_label": "PASS_MODERATE_V42A_ISOFLURANE_PASSES_CCL3F_EXTENSION_MIXED",
  "frozen_law": {
    "model": "scaled_atom_count",
    "q_s": 3.550419332185051,
    "q_p": 12.639790398730963,
    "B1": 1.0223463677280904,
    "B2": -0.9308578081435244,
    "gamma1": 1.0,
    "gamma2": 2.0,
    "descriptor": "atom_count",
    "reference_descriptor_value": 5.0
  },
  "target_descriptors": {
    "methane_reference": {
      "formula": "CH4",
      "atom_count": 5.0,
      "electron_count": 10.0,
      "heavy_atom_count": 1.0,
      "molar_mass": 16.04
    },
    "isoflurane": {
      "formula": "C3H2ClF5O",
      "atom_count": 12.0,
      "electron_count": 90.0,
      "heavy_atom_count": 10.0,
      "molar_mass": 184.49
    },
    "ccl3f": {
      "formula": "CCl3F",
      "atom_count": 5.0,
      "electron_count": 66.0,
      "heavy_atom_count": 5.0,
      "molar_mass": 137.37
    }
  },
  "res

## Figures

In [3]:
# Embedded figures from the V42A run. Re-run the full cell above to regenerate them.
print("Figures regenerated and exported by the full run cell.")

Figures regenerated and exported by the full run cell.
